# Lab 02 solution, notebook 1: dice with any number of sides, and a question that needs no tool

The worked answers to the two challenge tasks of `02a_meet_tool_calling.ipynb`. The setup repeats the notebook's
two tools, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import datetime
import json
import random

from openai import OpenAI, omit

client = OpenAI(timeout=60)

## Task 1: dice with any number of sides

The function first, called by hand. `range(count)` repeats the roll `count` times, and each roll is a number from 1
to `sides`.

In [3]:
def roll(sides: int, count: int) -> dict:
    """Roll `count` dice with `sides` sides each."""
    print(f"TOOL CALLED: roll(sides={sides}, count={count})")
    return {"rolls": [random.randint(1, sides) for _ in range(count)]}


print(roll(sides=6, count=2))

TOOL CALLED: roll(sides=6, count=2)
{'rolls': [1, 1]}


The schema has two properties, both of type `integer` and both required. The descriptions say what each number
means, because `sides` and `count` are easy to swap.

In [4]:
roll_tool = {
    "type": "function",
    "function": {
        "name": "roll",
        "description": "Roll several dice that all have the same number of sides.",
        "parameters": {
            "type": "object",
            "properties": {
                "sides": {"type": "integer", "description": "How many sides each die has."},
                "count": {"type": "integer", "description": "How many dice to roll."},
            },
            "required": ["sides", "count"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

The model asks. We print the call before anything runs.

In [5]:
messages = [{"role": "user", "content": "Roll three twenty-sided dice for me."}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[roll_tool],
    reasoning_effort=EFFORT or omit,
)

reply = response.choices[0].message
call = reply.tool_calls[0]

print("finish_reason:", response.choices[0].finish_reason)
print("name:         ", call.function.name)
print("arguments:    ", call.function.arguments)

finish_reason: tool_calls
name:          roll
arguments:     {"sides":20,"count":3}


`arguments` holds both values: 20 for `sides` and 3 for `count`. The model read "three" and "twenty-sided" in the
question and wrote them as whole numbers under the right keys. No `TOOL CALLED` line yet.

Now our code runs the function and sends the result back.

In [6]:
args = json.loads(call.function.arguments)
result = roll(**args)
print(result)

messages.append(reply)
messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[roll_tool],
    reasoning_effort=EFFORT or omit,
)

print(response.choices[0].message.content)

TOOL CALLED: roll(sides=20, count=3)
{'rolls': [2, 5, 11]}


You rolled **2, 5, and 11**.


One `TOOL CALLED` line, with `sides=20` and `count=3`. The three numbers in the reply (2, 5 and 11 in our run) are
the three numbers in `result`: they came from `random.randint` on our machine.

## Task 2: a question that needs no tool

The notebook's two tools again, both offered with a question that has nothing to do with dice or dates.

In [7]:
def roll_dice() -> dict:
    """Roll two six-sided dice."""
    print("TOOL CALLED: roll_dice()")
    return {"die_1": random.randint(1, 6), "die_2": random.randint(1, 6)}


def days_until(date: str) -> dict:
    """Count the days from today until a date written as YYYY-MM-DD."""
    print(f"TOOL CALLED: days_until({date})")
    target = datetime.date.fromisoformat(date)
    return {"date": date, "days": (target - datetime.date.today()).days}


dice_tool = {
    "type": "function",
    "function": {
        "name": "roll_dice",
        "description": "Roll two six-sided dice and return the two numbers.",
        "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False},
        "strict": True,
    },
}

days_tool = {
    "type": "function",
    "function": {
        "name": "days_until",
        "description": "Count the days from today until a date.",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "description": "The date to count to, as YYYY-MM-DD."}
            },
            "required": ["date"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

In [8]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of Jordan?"}],
    tools=[dice_tool, days_tool],
    reasoning_effort=EFFORT or omit,
)

reply = response.choices[0].message

print("finish_reason:", response.choices[0].finish_reason)
print("content:      ", reply.content)
print("tool_calls:   ", reply.tool_calls)

finish_reason: stop
content:       Amman.
tool_calls:    None


`finish_reason` is `stop`, `content` is the answer (Amman), and `tool_calls` is `None`. Neither function ran. Neither
description fits this question, and the model answered in text.

This is why code that handles a reply looks at `tool_calls` first. `reply.tool_calls[0]` on this reply would raise
`TypeError: 'NoneType' object is not subscriptable`.